# 19 — Uni-Mol Embeddings
3D-conformer-aware transformer embeddings from Uni-Mol (ICML 2023).
Pre-trained on 209M molecules with 3D coordinates.
Provides 512-dim CLS-token representation encoding 3D geometry — most
complementary to SMILES-sequence models.
Runtime: ~60 min (conformer generation is the bottleneck, CPU-only).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings, importlib.metadata
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from tqdm.auto import tqdm

from unimol_tools import UniMolRepr

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
SEED = 42
N_FOLDS = 5

print(f"unimol_tools version: {importlib.metadata.version('unimol-tools')}")
print(f"lightgbm version: {lgb.__version__}")
print("Setup complete.")

In [ ]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()

smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

print(f"Train: {len(smiles_tr):,}  |  Test: {len(smiles_te):,}")
print(f"y_tr range: {y_tr.min():.3f} – {y_tr.max():.3f}")

In [ ]:
# ── 3. Generate Uni-Mol embeddings ─────────────────────────────────────────────
CACHE_TR = DATA_PROCESSED / 'unimol_train_emb.npy'
CACHE_TE = DATA_PROCESSED / 'unimol_test_emb.npy'

def get_unimol_emb(smiles: list[str], cache: Path) -> np.ndarray:
    if cache.exists():
        print(f"  Loading cached embeddings from {cache.name}")
        return np.load(str(cache))
    clf = UniMolRepr(data_type='molecule', remove_hs=False)
    # get_repr returns a list of arrays (one per molecule), shape (512,) each
    reprs = clf.get_repr(smiles, return_atomic_reprs=False)
    X = np.array(reprs)          # (N, 512)
    np.save(str(cache), X.astype(np.float32))
    return X.astype(np.float32)

print("Generating train embeddings ...")
X_tr = get_unimol_emb(smiles_tr, CACHE_TR)
print(f"  Train: {X_tr.shape}")
print("Generating test embeddings ...")
X_te = get_unimol_emb(smiles_te, CACHE_TE)
print(f"  Test:  {X_te.shape}")
print(f"  NaN check — train: {np.isnan(X_tr).sum()}  test: {np.isnan(X_te).sum()}")

In [ ]:
# ── 4. Scaffold 5-fold CV with LGBM OOF ───────────────────────────────────────
LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.1, reg_lambda=0.1, n_jobs=4, verbose=-1)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof[va_idx] = m.predict(X_tr[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    met['fold'] = fold_i
    fold_metrics.append(met)
    print(f"  Fold {fold_i+1}: fold RAE={rae_fn(y_tr[va_idx], oof[va_idx]):.4f}  Spearman={met['Spearman']:.4f}")

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f"\nOOF RAE (Uni-Mol + LGBM): {oof_rae:.4f}")
print(f"Mean fold RAE:    {cv_df['RAE'].mean():.4f} +/- {cv_df['RAE'].std():.4f}")
print()
print("== Comparison ==")
print(f"  LGBM_base (Morgan + RDKit only): ~0.575")
print(f"  Chemprop multitask (nb 03):       0.517")
print(f"  Grand ensemble best:              0.5363")
print(f"  Uni-Mol + LGBM (this nb):        {oof_rae:.4f}")

np.save(DATA_PROCESSED / 'oof_unimol.npy', oof)

In [ ]:
# ── 5. Full retrain on all train data + predict test ──────────────────────────
final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_tr, y_tr)
te_preds = final_m.predict(X_te)
te_preds = np.clip(te_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_unimol.npy', te_preds)
print(f"Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}")

In [ ]:
# ── 6. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '19_unimol.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"OOF RAE (Uni-Mol LGBM): {oof_rae:.4f}")
print(f"Best ensemble RAE: 0.5363")
print(sub['pEC50'].describe().round(3))